# 분할 안정성 및 D4 특징 평균
사전 지정한 개발 데이터 3-fold × 5 seed. 확인 데이터 점수와 test 정답을 선택에 사용하지 않는다. 자세한 규칙은 EXPERIMENTS.md §11.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['OMP_NUM_THREADS']='2'
import json,time,warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC,SVC
from sklearn.metrics import f1_score,confusion_matrix,classification_report
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
from feature_helper import extract_features
thread_limit=threadpool_limits(limits=2)
train=pd.read_csv('data/train.csv');test=pd.read_csv('data/test.csv');sample=pd.read_csv('data/sample_submission.csv')
X=extract_features(train.image_path.tolist(),cache_path='train_features.npz')
Xt=extract_features(test.image_path.tolist(),cache_path='test_features.npz')
y=train.label.to_numpy();labels=[0,1,2,3,4]
dev,hold=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=42)
folds=list(StratifiedKFold(3,shuffle=True,random_state=42).split(X[dev],y[dev]))
assert len(dev)==600 and len(hold)==150 and not set(dev)&set(hold)
def macro(a,b):return float(f1_score(a,b,labels=labels,average='macro',zero_division=0))
candidates=[('LR_C1',LogisticRegression(C=1,max_iter=2000),0,1),
 ('LinearSVC_C0.1',LinearSVC(C=.1,dual='auto',max_iter=5000,random_state=42),1,.1),
 ('RBF_C1',SVC(C=1,gamma='scale',probability=False),2,1),
 ('LR_C10',LogisticRegression(C=10,max_iter=2000),0,10),
 ('LinearSVC_C1',LinearSVC(C=1,dual='auto',max_iter=5000,random_state=42),1,1),
 ('RBF_C10',SVC(C=10,gamma='scale',probability=False),2,10)]
def preprocessing():return make_pipeline(StandardScaler(),Normalizer())
def evaluate(model,features):
    t=time.perf_counter();pred=np.full(len(dev),-1,int);fs=[];fc=[];ws_all=[]
    for tr,va in folds:
        fit=clone(model)
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter('always');fit.fit(features[dev[tr]],y[dev[tr]])
        if any(issubclass(w.category,ConvergenceWarning) for w in ws):
            fit.set_params(**{list(fit.named_steps)[-1]+'__max_iter':fit.steps[-1][1].max_iter*2})
            with warnings.catch_warnings(record=True) as ws:
                warnings.simplefilter('always');fit.fit(features[dev[tr]],y[dev[tr]])
        ws_all.extend(str(w.message) for w in ws)
        assert not any(issubclass(w.category,ConvergenceWarning) for w in ws),'Unresolved convergence'
        pred[va]=fit.predict(features[dev[va]])
        fs.append(macro(y[dev[va]],pred[va]))
        fc.append(f1_score(y[dev[va]],pred[va],labels=labels,average=None,zero_division=0).tolist())
    return dict(score=macro(y[dev],pred),fold_scores=fs,fold_class_f1=fc,
      class_f1=f1_score(y[dev],pred,labels=labels,average=None,zero_division=0).tolist(),
      confusion=confusion_matrix(y[dev],pred,labels=labels).tolist(),
      seconds=time.perf_counter()-t,warnings=ws_all),pred


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


2026-10-01 19:09:25.510603336 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Using cached image features.
Using cached image features.


In [2]:
seeds=[42,7,19,73,101]
stability={};saved_oof={}
for C in [1,10]:
    reps=[]
    for seed in seeds:
        folds=list(StratifiedKFold(3,shuffle=True,random_state=seed).split(X[dev],y[dev]))
        result,pred=evaluate(make_pipeline(preprocessing(),LogisticRegression(C=C,max_iter=2000)),X)
        reps.append(result);saved_oof[f'C{C}_seed{seed}']=pred
    stability[str(C)]=reps
s1=np.array([a['score'] for a in stability['1']]);s10=np.array([a['score'] for a in stability['10']])
delta=s10-s1
selected_C=10 if delta.mean()>=.005 and (delta>0).sum()>=4 else 1
print('C=1:',s1,'C=10:',s10,'paired differences:',delta,'selected C:',selected_C)
robust=dict(seeds=seeds,stability=stability,stability_mean_delta=float(delta.mean()),selected_C=selected_C)
Path('robustness_results.json').write_text(json.dumps(robust,indent=2))


C=1: [0.94659957 0.95156909 0.94491574 0.9464768  0.94655309] C=10: [0.95815191 0.95660633 0.94977926 0.95659037 0.95317368] paired differences: [0.01155233 0.00503724 0.00486352 0.01011357 0.00662058] selected C: 10


15223

In [3]:
import hashlib,onnxruntime as ort
from datetime import datetime
from zoneinfo import ZoneInfo
encoder_path=Path('models/image_encoder.onnx')
opts=ort.SessionOptions();opts.intra_op_num_threads=2;opts.inter_op_num_threads=1
session=ort.InferenceSession(str(encoder_path),sess_options=opts,providers=['CPUExecutionProvider'])
input_name=session.get_inputs()[0].name
mean=np.array([.485,.456,.406],dtype=np.float32).reshape(3,1,1)
std=np.array([.229,.224,.225],dtype=np.float32).reshape(3,1,1)
def encode_image(im):
    a=np.asarray(im.convert('RGB').resize((224,224),Image.Resampling.BILINEAR),dtype=np.float32)/255
    x=np.ascontiguousarray(((a.transpose(2,0,1)-mean)/std)[None])
    out=session.run(None,{input_name:x})[0].reshape(-1)
    assert out.shape==(1280,) and np.isfinite(out).all()
    return out
transforms=[None,Image.Transpose.ROTATE_90,Image.Transpose.ROTATE_180,Image.Transpose.ROTATE_270]
def d4_features(paths,cache):
    h=hashlib.sha256(b'p05-d4-rawmean8-bilinear224-imagenet-v1')
    h.update(encoder_path.read_bytes())
    for path in paths:h.update(hashlib.sha256(Path(path).read_bytes()).digest())
    signature=h.hexdigest()
    if Path(cache).exists():
        with np.load(cache,allow_pickle=False) as d:
            if d['signature'].item()==signature and d['features'].shape==(len(paths),1280):
                return d['features'].copy()
    result=[]
    for i,path in enumerate(paths):
        with Image.open(path) as im:
            im=im.convert('RGB');vecs=[]
            for transform in transforms:
                view=im if transform is None else im.transpose(transform)
                vecs.extend([encode_image(view),encode_image(view.transpose(Image.Transpose.FLIP_LEFT_RIGHT))])
        result.append(np.mean(vecs,axis=0))
        if (i+1)%100==0:print('D4 processed',i+1,'/',len(paths),flush=True)
    result=np.asarray(result,dtype=np.float32)
    assert result.shape==(len(paths),1280) and np.isfinite(result).all()
    np.savez_compressed(cache,features=result,signature=signature)
    return result
with Image.open(train.image_path.iloc[dev[0]]) as im:
    identity_error=float(np.max(np.abs(encode_image(im)-X[dev[0]])))
assert identity_error<1e-5
print('Identity encoder max abs difference:',identity_error)
t=time.perf_counter()
# Compute development views first; holdout remains unscored.
Xd=d4_features(train.iloc[dev].image_path.tolist(),'dev_d4_features.npz')
Xaug=np.zeros_like(X);Xaug[dev]=Xd
aug_results=[]
for seed in seeds:
    folds=list(StratifiedKFold(3,shuffle=True,random_state=seed).split(X[dev],y[dev]))
    result,pred=evaluate(make_pipeline(preprocessing(),LogisticRegression(C=selected_C,max_iter=2000)),Xaug)
    aug_results.append(result);saved_oof[f'D4_seed{seed}']=pred
base=stability[str(selected_C)]
difference=np.array([a['score']-b['score'] for a,b in zip(aug_results,base)])
class_delta=np.mean([a['class_f1'] for a in aug_results],axis=0)-np.mean([a['class_f1'] for a in base],axis=0)
criteria=dict(mean_gain_005=bool(difference.mean()>=.005),positive_4_of_5=bool((difference>0).sum()>=4),
 river_improves=bool(class_delta[4]>0),no_class_drop_003=bool(class_delta.min()>=-.03))
adopt=all(criteria.values())
robust.update(d4=aug_results,d4_deltas=difference.tolist(),d4_class_delta=class_delta.tolist(),
 d4_criteria=criteria,d4_adopted=adopt,identity_max_error=identity_error,d4_seconds=time.perf_counter()-t)
print('D4 differences',difference,'class differences',class_delta,'adopt:',adopt,criteria)
np.savez_compressed('robustness_oof.npz',dev=dev,**saved_oof)
Path('robustness_results.json').write_text(json.dumps(robust,indent=2))


Identity encoder max abs difference: 0.0


D4 processed 100 / 600


D4 processed 200 / 600


D4 processed 300 / 600


D4 processed 400 / 600


D4 processed 500 / 600


D4 processed 600 / 600


D4 differences [-0.00670947  0.00159204  0.00164946 -0.00666151  0.00175796] class differences [ 1.46711713e-05  8.99928577e-05 -8.97851123e-04 -6.46010230e-03
 -1.11822952e-03] adopt: False {'mean_gain_005': False, 'positive_4_of_5': False, 'river_improves': False, 'no_class_drop_003': True}


22563

In [4]:
if adopt:
    Xfinal=np.empty_like(X);Xfinal[dev]=Xd
    Xfinal[hold]=d4_features(train.iloc[hold].image_path.tolist(),'hold_d4_features.npz')
    Xtest=d4_features(test.image_path.tolist(),'test_d4_features.npz')
    model=make_pipeline(preprocessing(),LogisticRegression(C=selected_C,max_iter=2000))
    model.fit(Xfinal,y)
    prediction=model.predict(Xtest).astype(int)
    sub=sample[['image_id']].copy()
    sub['label']=sub.image_id.map(pd.Series(prediction,index=test.image_id)).astype(int)
    assert sub.image_id.equals(sample.image_id) and sub.image_id.is_unique and len(sub)==250
    assert not sub.isna().any().any() and sub.label.isin(labels).all()
    sub.to_csv('submission_codex_d4.csv',index=False)
    pd.testing.assert_frame_equal(sub,pd.read_csv('submission_codex_d4.csv'))
    counts={}
    for filename in ['submission_codex.csv','submission_codex_posthoc.csv']:
        old=pd.read_csv(filename).set_index('image_id').loc[sub.image_id,'label'].to_numpy()
        counts[filename]=int((old!=sub.label.to_numpy()).sum())
    robust['prediction_changes']=counts
    robust['submission']='submission_codex_d4.csv'
    print('Saved',robust['submission'],'changed:',counts)
else:
    robust['submission']=None
    print('D4 not adopted. Existing submissions retained.')
robust['finished']=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
Path('robustness_results.json').write_text(json.dumps(robust,indent=2))


D4 not adopted. Existing submissions retained.


22635